In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:

class Recommendation:
    buy = 2
    sell = 3
    neutral = 1

class DataPreProcessing:
    
    def __init__(self) -> pd.DataFrame:
        pass

    
    def get_data(self,asset: str,timeframe: int) -> pd.DataFrame:
        data_pd = pd.read_csv(f'backend/dolphin/Data/{asset}_{timeframe}_MIN.csv')
        return data_pd
    
    def add_Indicators_column(self, columns: list=[]) -> None:
        if columns is None:
            columns = ['RSI','MCAD','W%R','CCI', 'ADX','STOCH','AO']
        self.Df_data[columns] = None

    def RSI(rsi):
        """Compute Relative Strength Index

        Args:
            rsi (float): RSI value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        if (rsi < 30):
            return Recommendation.buy
        elif (rsi > 70):
            return Recommendation.sell
        else:
            return Recommendation.neutral

    def Stoch(k, d):
        """Compute Stochastic

        Args:
            k (float): Stoch.K value
            d (float): Stoch.D value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        if (k < 20 and d < 20 and k > d):
            return Recommendation.buy
        elif (k > 80 and d > 80 and k < d):
            return Recommendation.sell
        else:
            return Recommendation.neutral

    def CCI20(cci20,rsi):
        """Compute Commodity Channel Index 20

        Args:
            cci20 (float): CCI20 value
            cci201 ([type]): CCI20[1] value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        rsi = DataPreProcessing.RSI(rsi)
        if (cci20 < -100) and (rsi == 2):
            return Recommendation.buy
        elif (cci20 > 100) and (rsi == 1):
            return Recommendation.sell
        else:
            return Recommendation.neutral

    def ADX(adx, adxpdi, adxndi):
        """Compute Average Directional Index

        Args:
            adx (float): ADX value
            adxpdi (float): ADX+DI value
            adxndi (float): ADX-DI value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        if (adx > 20 and adxpdi > adxndi):
            return Recommendation.buy
        elif (adx > 20 and adxpdi < adxndi):
            return Recommendation.sell
        else:
            return Recommendation.neutral

    def AO(ao,rsi):
        """Compute Awesome Oscillator

        Args:
            ao (float): AO value
            ao1 (float): AO[1] value
            ao2 (float): AO[2] value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        rsi = DataPreProcessing.RSI(rsi)
        if (ao > 0) and rsi == 2:
            return Recommendation.buy
        elif (ao < 0) and rsi == 3:
            return Recommendation.sell
        else:
            return Recommendation.neutral
    
    def UO(uo,rsi):
        """Compute Awesome Oscillator

        Args:
            uo (float): UO value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        rsi = DataPreProcessing.RSI(rsi)
        if (uo > 0) and rsi == 2:
            return Recommendation.buy
        elif (uo < 0) and rsi == 3:
            return Recommendation.sell
        else:
            return Recommendation.neutral
    
    def MACD(macd, signal):
        """Compute Moving Average Convergence/Divergence

        Args:
            macd (float): MACD.macd value
            signal (float): MACD.signal value

        Returns:
            string: "BUY", "SELL", or "NEUTRAL"
        """
        if (macd > signal):
            return Recommendation.buy
        elif (macd < signal):
            return Recommendation.sell
        else:
            return Recommendation.neutral
        
    def WR(wr,rsi):
        rsi = DataPreProcessing.RSI(rsi)
        if wr > -20 and rsi == 3:
            return Recommendation.sell
        elif wr < -80 and rsi == 2:
            return Recommendation.buy
        else:
            return Recommendation.neutral

In [ ]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [ ]:

DATA_PATH = "EURUSD_5_MIN.csv"
data = pd.read_csv(DATA_PATH)
# data = add_all_ta_features(data,'open','close','low','high','volume')


In [ ]:
data_1=ta.add_trend_ta(data,'high','low','close')
data_2=ta.add_momentum_ta(data,'high','low','close','volume')
data_3=ta.add_volatility_ta(data,'high','low','close','volume')
from ta.trend import macd,cci,adx
from ta.momentum import rsi
rsi = rsi(data['close'],14)
rsi.dropna(axis=0,inplace=True)
cci = cci(data['high'],data['low'],data['close'],14)
cci.dropna(axis=0,inplace=True)
adx = adx(data['high'],data['low'],data['close'])
adx.dropna(axis=0,inplace=True)
new_data = data[14:]
new_data['rsi'] = rsi
new_data['cci'] = cci
new_data['adx'] = adx.iloc[13:]
print(new_data.columns)

In [ ]:
# indicator_list = ['datetime','symbol','high','low','open','close','volume','trend_macd','trend_macd_signal','trend_adx', 'trend_adx_pos', 'trend_adx_neg', 'trend_cci','momentum_rsi',
#                                'momentum_stoch', 'momentum_stoch_signal', 'momentum_uo','momentum_wr', 'momentum_ao']
indicator_list = ['datetime','high','low','open','close','volume','rsi','cci','adx','trend_adx_pos','trend_adx_neg','trend_macd','trend_macd_signal','momentum_wr','volatility_bbh','volatility_bbl','momentum_stoch','momentum_stoch_signal']
data = new_data[indicator_list]
print(data.head())

In [ ]:
data['stoch'] = np.NaN
for index, row in data.iterrows():
    momentum_stoch = row['momentum_stoch']
    momentum_stoch_signal = row['momentum_stoch_signal']
    if (momentum_stoch < 20 and momentum_stoch_signal < 20 and momentum_stoch > momentum_stoch_signal):
        data.loc[index, 'stoch'] = 1
    elif (momentum_stoch > 80 and momentum_stoch_signal > 80 and momentum_stoch < momentum_stoch_signal):
        data.loc[index, 'stoch'] = 2
    else:
        data.loc[index, 'stoch'] = 0

In [ ]:
data['macd'] = np.NaN
for index, row in data.iterrows():
    trend_macd = row['trend_macd']
    trend_macd_signal = row['trend_macd_signal']
    data.loc[index, 'macd'] = 0
    if trend_macd > 0 and trend_macd_signal > 0:
        if (trend_macd < trend_macd_signal):
            data.loc[index, 'macd'] = 2
    elif trend_macd < 0 and trend_macd_signal < 0:
        if (trend_macd > trend_macd_signal):
            data.loc[index, 'macd'] = 1
    else:
        data.loc[index, 'macd'] = 0

In [ ]:
data['BBP'] = np.NaN
for index, row in data.iterrows():
    bbh_value = row['volatility_bbh']
    bbl_value = row['volatility_bbl']
    if bbh_value < row['close']:
        data.loc[index, 'BBP'] = 1
    elif bbh_value > row['close']:
        data.loc[index, 'BBP'] = 2
    else:
        data.loc[index, 'BBP'] = 0

In [ ]:
for index, row in data.iterrows():
    adx_value = row['adx']
    data.loc[index, 'adx'] = 0
    if adx_value > 40 or (20 < adx_value < 40):
        if row['trend_adx_pos'] > row['trend_adx_neg']:
            data.loc[index, 'adx'] = 1
        elif row['trend_adx_pos'] < row['trend_adx_neg']:
            data.loc[index, 'adx'] = 2
        else:
            data.loc[index, 'adx'] = 0

In [ ]:
data['W%R'] = np.NaN
for index, row in data.iterrows():
    wr_value = row['momentum_wr']
    data.loc[index, 'W%R'] = 0
    if wr_value < -80:
        data.loc[index, 'W%R'] = 1
    elif wr_value > -20:
        data.loc[index, 'W%R'] = 2
    else:
        data.loc[index, 'W%R'] = 0

In [ ]:
data['Prediction'] = np.NaN
for index, row in data.iterrows():
    open_val = row['open']
    close_val = row['close']
    if open_val > close_val:
        data.loc[index, 'Prediction'] = 2
    elif open_val < close_val:
        data.loc[index, 'Prediction'] = 1
    else:
        data.loc[index, 'Prediction'] = 0

In [ ]:

# Define conditions and choices
conditions = [
    data['open'] == data['close'],
    data['open'] < data['close']
]
rsi_conditions = [
    lambda x: x > 75,
    lambda x: x < 30
]
cci_conditions = [
    lambda x: x < -190,
    lambda x: x > 190
]
choices = [0, 1]
# Apply conditions using np.select()
# data['Prediction'] = np.select(conditions, choices, default=2)
rsi_choices = [1,2]
cci_choices = [1,2]
data['rsi'] = data['rsi'].apply(lambda x: next((c for c, cond in zip(rsi_choices, rsi_conditions) if cond(x)), 0))
data['cci'] = data['cci'].apply(lambda x: next((c for c, cond in zip(cci_choices, cci_conditions) if cond(x)), 0))
data['adx'] = data['adx'].astype(int)
data['W%R'] = data['W%R'].astype(int)
# data['BBP'] = data['BBP'].astype(int)
data['stoch'] = data['stoch'].astype(int)
data['macd'] = data['macd'].astype(int)
data['Prediction'] = data['Prediction'].astype(int)
data.drop(columns=['trend_adx_pos','trend_adx_neg','momentum_wr','volatility_bbh','volatility_bbl','momentum_stoch','momentum_stoch_signal','trend_macd_signal','trend_macd'],inplace=True)
data['rsi_1'] = np.select(conditions, rsi_choices, default=0)

print(data.shape)
print(data.head())

In [ ]:
# Define conditions for setting 'Prediction' column to 0
# conditions = (data['W%R'] == 0) & (data['macd'] == 0) & (data['rsi'] == 0) & (data['cci'] == 0) & (data['stoch'] == 0) & (data['adx'] == 0)

# Use loc to set 'Prediction' to 0 where conditions are met
# data.loc[conditions, 'Prediction'] = 0
# data.drop(index=data[(data['W%R'] == 0) & (data['macd'] == 0) & (data['rsi'] == 0) & (data['cci'] == 0) & (data['stoch'] == 0) & (data['adx'] == 0)].index, inplace=True)

In [ ]:

data.iloc[:,7:-1].replace(0.0,pd.NA,inplace=True)
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.head())
print(data.shape)


In [ ]:
data['trend_macd'] = data.apply(lambda x: DataPreProcessing.MACD(x['trend_macd'],x['trend_macd_signal']),axis=1)
data['trend_adx'] = data.apply(lambda x: DataPreProcessing.ADX(x['trend_adx'],x['trend_adx_pos'],x['trend_adx_neg']),axis=1)
data['trend_cci'] = data.apply(lambda x: DataPreProcessing.CCI20(x['trend_cci'],x['momentum_rsi']),axis=1)
data['momentum_rsi'] = data.apply(lambda x: DataPreProcessing.RSI(x['momentum_rsi']),axis=1)
data['momentum_stoch'] = data.apply(lambda x: DataPreProcessing.Stoch(x['momentum_stoch'],x['momentum_stoch_signal']),axis=1)
data['momentum_ao'] = data.apply(lambda x: DataPreProcessing.AO(x['momentum_ao'],x['momentum_rsi']),axis=1)
data['momentum_uo'] = data.apply(lambda x: DataPreProcessing.AO(x['momentum_uo'],x['momentum_rsi']),axis=1)
data['momentum_wr'] = data.apply(lambda x: DataPreProcessing.WR(x['momentum_wr'],x['momentum_rsi']),axis=1)
data.drop(['trend_adx','trend_macd','momentum_stoch','momentum_ao','momentum_uo','momentum_stoch_signal','trend_adx_pos','trend_adx_neg','trend_adx_pos','trend_adx_neg','trend_macd_signal'],axis=1,inplace=True)
print(data.columns)
print(data.head())
data.to_csv('new_output.csv')

In [ ]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


In [ ]:

X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.head())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.2, random_state = 24)



In [ ]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier()
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


In [ ]:

# # Defining scoring metric for k-fold cross validation
# def cv_scoring(estimator, X, y):
#     return accuracy_score(y, estimator.predict(X))
 
# # Initializing Models
# models = {
#     "SVC":SVC(),
#     "Random Forest":RandomForestClassifier(random_state=18),
#     "KNN":KNeighborsClassifier(n_neighbors=3)
# }
 
# # Producing cross validation score for the models
# for model_name in models:
#     model = models[model_name]
#     scores = cross_val_score(model, X, y, cv = 10, 
#                              n_jobs = -1, 
#                              scoring = cv_scoring)
#     print("=="*30)
#     print(model_name)
#     print(f"Scores: {scores}")
#     print(f"Mean Score: {np.mean(scores)}")


In [ ]:

svm_model = SVC()
svm_model.fit(X_train, y_train)
preds = svm_model.predict(X_test)
 
print(f"Accuracy on train data by SVM Classifier\
: {accuracy_score(y_train, svm_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by SVM Classifier\
: {accuracy_score(y_test, preds)*100}")


In [ ]:

# cf_matrix = confusion_matrix(y_test, preds)
# plt.figure(figsize=(12,8))
# sns.heatmap(cf_matrix, annot=True)
# plt.title("Confusion Matrix for SVM Classifier on Test Data")
# plt.show()

In [ ]:
# Training and testing Random Forest Classifier
rf_model = RandomForestClassifier(random_state=18)
rf_model.fit(X_train, y_train)
preds = rf_model.predict(X_test)
print(f"Accuracy on train data by Random Forest Classifier\
: {accuracy_score(y_train, rf_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by Random Forest Classifier\
: {accuracy_score(y_test, preds)*100}")
 


In [ ]:
# cf_matrix = confusion_matrix(y_test, preds)
# plt.figure(figsize=(12,8))
# sns.heatmap(cf_matrix, annot=True)
# plt.title("Confusion Matrix for Random Forest Classifier on Test Data")
# plt.show()

In [ ]:
kn_model = KNeighborsClassifier(n_neighbors=3)
kn_model.fit(X_train, y_train)
preds = kn_model.predict(X_test)
print(f"Accuracy on train data by K Neighbors Classifier\
: {accuracy_score(y_train, kn_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by K Neighbors Classifier\
: {accuracy_score(y_test, preds)*100}")

In [ ]:
# cf_matrix = confusion_matrix(y_test, preds)
# plt.figure(figsize=(12,8))
# sns.heatmap(cf_matrix, annot=True)
# plt.title("Confusion Matrix for Random Forest Classifier on Test Data")
# plt.show()

In [ ]:
# Training the models on whole data
final_svm_model = SVC()
final_rf_model = RandomForestClassifier(random_state=18)
final_kn_model = KNeighborsClassifier(n_neighbors=3)
final_xgb_model = XGBClassifier()
final_svm_model.fit(X, y)
final_rf_model.fit(X, y)
final_kn_model.fit(X, y)
final_xgb_model.fit(X, y)
 

In [ ]:
import requests
response = requests.get("http://localhost:8001/signal/")
data = response.json()
print(data)
# 'rsi', 'cci',
# 'adx', 'trend_adx_pos', 'trend_adx_neg', 'trend_macd',
# 'trend_macd_signal', 'momentum_wr', 'volatility_bbh', 'volatility_bbl',
# 'momentum_stoch', 'momentum_stoch_signal',
"""
dict_keys(['Recommend.Other', 'Recommend.All', 'Recommend.MA', 'RSI', 'RSI[1]', 'Stoch.K', 'Stoch.D', 'Stoch.K[1]', 'Stoch.D[1]', 
'CCI20', 'CCI20[1]', 'ADX', 'ADX+DI', 'ADX-DI', 'ADX+DI[1]', 'ADX-DI[1]', 'AO', 'AO[1]', 'Mom', 'Mom[1]', 'MACD.macd', 'MACD.signal',
'Rec.Stoch.RSI', 'Stoch.RSI.K', 'Rec.WR', 'W.R', 'Rec.BBPower', 'BBPower', 'Rec.UO', 'UO', 'close', 'EMA5', 'SMA5', 'EMA10', 'SMA10', 
'EMA20', 'SMA20', 'EMA30', 'SMA30', 'EMA50', 'SMA50', 'EMA100', 'SMA100', 'EMA200', 'SMA200', 'Rec.Ichimoku', 'Ichimoku.BLine', 'Rec.VWMA', 
'VWMA', 'Rec.HullMA9', 'HullMA9', 'Pivot.M.Classic.S3', 'Pivot.M.Classic.S2', 'Pivot.M.Classic.S1', 'Pivot.M.Classic.Middle', 'Pivot.M.Classic.R1', 
'Pivot.M.Classic.R2', 'Pivot.M.Classic.R3', 'Pivot.M.Fibonacci.S3', 'Pivot.M.Fibonacci.S2', 'Pivot.M.Fibonacci.S1', 'Pivot.M.Fibonacci.Middle', 
'Pivot.M.Fibonacci.R1', 'Pivot.M.Fibonacci.R2', 'Pivot.M.Fibonacci.R3', 'Pivot.M.Camarilla.S3', 'Pivot.M.Camarilla.S2', 'Pivot.M.Camarilla.S1', 
'Pivot.M.Camarilla.Middle', 'Pivot.M.Camarilla.R1', 'Pivot.M.Camarilla.R2',
'Pivot.M.Camarilla.R3', 'Pivot.M.Woodie.S3', 'Pivot.M.Woodie.S2', 'Pivot.M.Woodie.S1', 'Pivot.M.Woodie.Middle', 'Pivot.M.Woodie.R1', 
'Pivot.M.Woodie.R2', 'Pivot.M.Woodie.R3', 'Pivot.M.Demark.S1', 'Pivot.M.Demark.Middle', 'Pivot.M.Demark.R1', 'open', 
'P.SAR', 'BB.lower', 'BB.upper', 'AO[2]', 'volume', 'change', 'low', 'high'])
"""
input_data = {
    'rsi':[data['RSI']],
    'cci': [data['CCI20']],
    'adx':[data['ADX']],
    'trend_adx_pos':[data['ADX+DI']],
    'trend_adx_neg':[data['ADX-DI']],
    'trend_macd': [data['MACD.macd']],
    'trend_macd_signal': [data['MACD.signal']],
    'momentum_wr': [data['W.R']],
    'volatility_bbh':[data['BB.lower']],
    'volatility_bbl':[data['BB.upper']],
    'momentum_stoch':[data['Stoch.K']],
    'momentum_stoch_signal':[data['Stoch.D']]    
}
# rsi  cci  adx  stoch  W%R
# input_data = {
#     'rsi': [prediction.get(data['COMPUTE']['RSI'])],
#     'cci': [prediction.get(data['COMPUTE']['CCI'])],
#     'adx': [prediction.get(data['COMPUTE']['ADX'])],
#     'stoch': [prediction.get(data['COMPUTE']['STOCH.K'])],
#     'macd': [prediction.get(data['COMPUTE']['MACD'])],
#     'W%R': [prediction.get(data['COMPUTE']['W%R'])]
    
# }
print(input_data)
data = pd.DataFrame(input_data)
rf = final_rf_model.predict(data)
svm = final_svm_model.predict(data)
knn = final_kn_model.predict(data)
xgb = final_xgb_model.predict(data)
print(rf,svm,knn,xgb)
# DATA_PATH = "EURUSD_5_MIN.csv"
# data = pd.read_csv(DATA_PATH)
# indicator_list = ['datetime','symbol','high','low','open','close','volume', 'trend_cci','momentum_rsi',
#                     'momentum_uo','momentum_wr']
# data = add_all_ta_features(data,'open','close','low','high','volume')
# data = data[indicator_list]


In [ ]:

# data.replace(0.0,pd.NA,inplace=True)
# data.dropna(inplace=True)
# data.reset_index(drop=True,inplace=True)
# # Define conditions and choices
# conditions = [
#     data['open'] == data['close'],
#     data['open'] < data['close']
# ]
# choices = ['NEUTRAL', 'BUY']
# # Apply conditions using np.select()
# data['Prediction'] = np.select(conditions, choices, default='SELL')
# print(data.head())

In [ ]:
# data_1 = data.iloc[:,7:-1]
# print(data_1.columns)
# rf = final_rf_model.predict(data_1)
# svm = final_svm_model.predict(data_1)
# knn = final_kn_model.predict(data_1)
# data['support vector'] = svm
# data['random_forest'] = rf
# data['knn'] = knn
# data.to_csv('final_output.csv')